# Session 6: Advanced RAG - Embeddings, Vector DBs, Knowledge Bases

In [ ]:
from src.rag.pipeline import RAGPipeline
from src.rag.vector_store import VectorStore
from src.rag.retrieval import query_expansion, rrf_fusion
from src.utils.llm import get_llm
from langchain_core.documents import Document

llm = get_llm()
print("Advanced RAG ready")

## 1. Query Expansion

In [ ]:
query = "What are Apple's growth prospects?"
expanded = query_expansion(query, llm)
print("Expanded queries:")
for q in expanded:
    print(f"  - {q}")

## 2. RRF (Reciprocal Rank Fusion)

In [ ]:
list_a = [Document("Apple services revenue growing", metadata={"id": "d1"}), Document("iPhone sales trends", metadata={"id": "d2"})]
list_b = [Document("iPhone sales trends", metadata={"id": "d2"}), Document("Apple services revenue growing", metadata={"id": "d1"}), Document("MacBook market share", metadata={"id": "d3"})]

fused = rrf_fusion([list_a, list_b])
print("Fused results:")
for d in fused:
    print(f"  {d.page_content}")

## 3. Embedding Visualization

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
companies = ["Apple", "Microsoft", "Google", "Amazon", "Tesla", "Ford", "Goldman Sachs", "JPMorgan"]
embeddings = model.encode(companies)

# Pairwise similarity
sim = np.dot(embeddings, embeddings.T)
print("Company similarity matrix:")
for i, c1 in enumerate(companies):
    for j, c2 in enumerate(companies):
        if i < j:
            print(f"  {c1:15s} <-> {c2:15s}: {sim[i][j]:.3f}")

## 4. Full RAG Pipeline Run

In [ ]:
store = VectorStore()
docs = [
    Document("AAPL: Strong services growth offsetting iPhone maturity. Revenue $120B, Services $25B.", metadata={"ticker": "AAPL"}),
    Document("MSFT: Azure accelerating, AI integration driving enterprise deals. Revenue $65B.", metadata={"ticker": "MSFT"}),
    Document("Sector outlook: Tech valuations elevated but AI tailwinds support premium multiples.", metadata={"topic": "outlook"}),
]
store.add_documents(docs)

pipeline = RAGPipeline(store, docs)
results = pipeline.retrieve("Apple revenue and growth drivers", llm=llm)
print("Top results:")
for d in results:
    print(f"  [{d.metadata}] {d.page_content}")